# STR01 · 미션 09 | AAR 기반 결핍 경보 지연 원인 분석 및 경보 임계 파라미터 재보정

**학생용 Student · 직접 실습**  
CMS 편성: 3시간 · 교육용 가상 데이터 사용

## 1. 상황과 이번 미션의 문제

가상의 20개 조직과 5종 물품의 기록을 이용해 **AAR 기반 결핍 경보 지연 원인 분석 및 경보 임계 파라미터 재보정** 과제를 수행합니다. 실제 조직의 기록이 아닌 실습자료이며, 코드를 실행한 결과도 실제 현장 상태를 뜻하지 않습니다.

**이번 미션에서 해결할 문제:** 실제 결핍(재고 0 진입) 사건 48건에 대해 기존 단순 DOS 경보의 정시·지연·미탐을 분류하고 원인을 템포 급변, 추정 오차, 임계 과소로 나눈다. 예측 기반 경보의 임계를 보정 기간에서 격자 탐색으로 정하고 검증 기간에서 확인한다. 비용 절감액, 시간 단축률, 결핍 사건 감소율의 3대 효과 지표를 가정 단가와 산식으로 산출한다.

**만들 결과물:** `reports/str01_roi_analysis.md`, `src/analysis/str01_aar.py`, `outputs/M09/G.csv`, `forms/STR01_M09_forms.md`

## 2. 처음 시작하는 방법

1. 아래 **환경 준비 셀**을 실행합니다. Colab에서는 메뉴의 **런타임 → 모두 실행** 대신 각 셀의 결과를 먼저 확인하면서 진행해도 됩니다.
2. 핵심 함수의 `TODO`를 AI와 함께 구현합니다. 처음에는 함수의 입력·출력과 왜 필요한지 질문하세요.
3. 첫 실행 결과를 확인하고 실패하면 오류 메시지를 AI에게 설명해 달라고 요청합니다.
4. AI가 제안한 코드 중 최소 3가지를 원자료와 실행 결과로 대조하고 수정 이유를 기록합니다.
5. 활동별 작성 칸을 채운 뒤 **실제 AI 대화 원문**을 보존하고 마지막 저장 셀로 ZIP을 만듭니다.

**보안과 평가:** 실제 조직·위치·재고·작전 자료, 개인정보를 외부 AI에 입력하지 않습니다. 모델 성능은 **실제 출력 로그**로 판단하며, 예시값을 실측으로 적지 않습니다. 실행 성공이나 ZIP 저장은 학습 PASS·실제 승인과 다릅니다. 폐쇄망에서는 허용된 Python/Jupyter 환경에서 실행하세요. 인터넷이 필요한 `pip` 설치는 폐쇄망에서 실패할 수 있으므로 사전 패키지 준비가 필요합니다.


In [ ]:
# @title ① 필수 실습자료 ZIP 업로드 및 환경 준비 (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 업로드 파일명은 자유이며, .zip 확장자와 유효한 ZIP 내용만 확인합니다.
WORKSPACE = (Path.cwd() / 'STR01_실습프로젝트_작업공간').resolve()
STR01_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global STR01_PROJECT_READY, STR01_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        archive.extractall(WORKSPACE)
    os.chdir(WORKSPACE)
    STR01_PROJECT_DIR = WORKSPACE
    STR01_PROJECT_READY = True
    print('✅ 실습자료 압축 해제 완료:', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

print('【필수】실습 시작 전 프로젝트 실습자료 ZIP 파일을 업로드해야 합니다.')
print('ZIP 파일명은 자유입니다. 확장자가 .zip인 파일을 선택하세요.')
try:
    import google.colab
    from google.colab import files as colab_files
    print('① [파일 선택]에서 실습자료 .zip 파일을 한 개 선택하세요.')
    incoming = colab_files.upload()
    zip_items = [(name, data) for name, data in incoming.items() if name.lower().endswith('.zip')]
    if len(zip_items) != 1 or len(incoming) != 1:
        raise RuntimeError('ZIP 파일 한 개만 업로드해 주세요. 이 셀을 다시 실행하세요.')
    selected_name, selected_data = zip_items[0]
    print('선택한 ZIP:', selected_name)
    _prepare_project_zip(selected_data)
except ImportError:
    # 로컬 Jupyter: 파일명이 달라도 .zip 확장자이면 업로드할 수 있습니다.
    import ipywidgets as widgets
    from IPython.display import display
    uploader = widgets.FileUpload(accept='.zip', multiple=False, description='실습 ZIP 업로드')
    upload_log = widgets.Output()
    def _on_upload(change):
        with upload_log:
            upload_log.clear_output()
            try:
                values = uploader.value
                items = list(values.values()) if isinstance(values, dict) else list(values)
                if len(items) != 1:
                    print('❌ ZIP 파일 한 개를 선택해 주세요.')
                    return
                selected = items[0]
                filename = selected['name']
                if not filename.lower().endswith('.zip'):
                    print('❌ .zip 확장자 파일만 선택할 수 있습니다.')
                    return
                print('선택한 ZIP:', filename)
                _prepare_project_zip(bytes(selected['content']))
            except Exception as exc:
                print('❌ 업로드/압축 해제 실패:', exc)
    uploader.observe(_on_upload, names='value')
    display(Markdown('**필수:** 프로젝트 실습자료가 들어 있는 `.zip` 파일을 선택하세요. 파일명은 자유이며, **압축 해제 완료** 문구를 확인한 뒤 다음 셀로 이동합니다.'))
    display(uploader, upload_log)
    print('⏳ 실습 ZIP 업로드 대기 중입니다.')
print('실행 환경: Python', sys.version.split()[0])


## 미션 핵심 개념과 원본 실습 설명

---

## 🎯 미션 09. AAR 기반 결핍 경보 지연 원인 분석 및 경보 임계 파라미터 재보정
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 결핍 경보 지연 원인을 분석하고 임계 파라미터를 재보정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **AAR 사례의 결핍 경보 지연 원인을 분석하고 경보 임계 파라미터를 재보정한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **실제 결핍(재고 0 진입) 사건 48건에 대해 기존 단순 DOS 경보의 정시·지연·미탐을 분류하고 원인을 템포 급변, 추정 오차, 임계 과소로 나눈다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `reports/str01_roi_analysis.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추정 산식의 객관성 및 군 재정/군수 기획 부서 설득력

#### 미션 PASS 조건

- 정량 기준: 비용 절감액, 시간 단축률 등 3대 정량 효과 지표 산출


- 진행 기준: ROI 분석 보고서 완성 및 주요 지표 민감도 분석 수록


- 검증 방법: 회계/기획 기준 모델과의 비교 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



### AI 활용 예시 · 미션 09 전용

> 나는 Python을 처음 배우는 학습자야. **AAR 기반 결핍 경보 지연 원인 분석 및 경보 임계 파라미터 재보정**을 진행하고 있어. 다음 요구사항을 보고 핵심 함수의 입력·출력, 필요한 검증 3개를 쉬운 말로 설명해줘. `TODO` 구간만 구현할 수 있는 코드를 제안하고, 데이터를 미리 보지 않는 검증 방법과 경계 사례도 알려줘. 실행 결과는 만들지 말고 내가 실제 실행한 로그를 받은 다음 분석해줘.  
> 요구사항: src/analysis/str01_aar.py / first_alert: u,it,t_e,th,col → 사건 전 최초 경보일 또는 None.95～110 보정 사건은95일부터,111～125 검증 사건은111일부터 탐색한다. 정답을 알기 전에 이미 저장된 경보 자료만 사용하고 사건일 이후는 보지 않는다.

**AI 답변을 받으면:** 그대로 복사하지 말고 입력 시점·자료 범위·실제 실행 여부를 직접 확인하세요. 확인되지 않은 수치는 `미측정`, 수행하지 않은 활동은 `미실시`로 남깁니다.


In [ ]:
if not globals().get('STR01_PROJECT_READY', False):
    raise RuntimeError('실습 전에 ① 환경 설정 셀에서 실습자료 ZIP 파일을 업로드하세요.')

"""AAR 사례에서 결핍 경보가 늦게 나간 원인(임계 과소, 템포 급변 미반영, 예측 오차)을 분류하고, 경보 임계 파라미터를 보정 기간에서 재보정해 검증 기간에서 선행 시간과 오탐률을 비교하며, 비용 절감액·시간 단축률·결핍 사건 감소율의 3대 정량 효과 지표를 가정과 산식으로 산출한다."""
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 부대 20개 x 핵심 보급품 5종 일일 재고·소모 시뮬레이터 (실제 부대·수량·기준일수 아님)
# 핵심성과지표(KPI) = 기준 보급일수(DOS) = 현재 재고 / 최근 평균 일일 소모량. 목표 DOS는 교육용 가정값이다.
# 결핍 위험 일수 H = 보급이 없다고 가정할 때 현재 재고가 바닥나기까지 남은 일수(미래 실제 소모량으로 계산한 정답).
# ---------------------------------------------------------------------------
ITEMS = {"식량": (100, 0.3, 10), "피복": (20, 0.1, 15), "유류": (300, 1.2, 7), "탄약": (60, 1.5, 6), "수리부속": (15, 0.5, 12)}   # (기본 일소모, 템포 민감도, 목표 DOS)
N_UNIT, DAYS = 20, 160
UNITS = [f"가상{u:02d}부대" for u in range(1, N_UNIT + 1)]


def make_data(seed=1, n_unit=N_UNIT, days=DAYS):
    """일 단위 재고 원장: unit, item, day, tempo(실제 작전 템포), plan(사전 계획 템포), wx(악천후), consume, resupply, stock."""
    rng = np.random.default_rng(seed)
    rows = []
    for u in range(1, n_unit + 1):
        s, ph = rng.uniform(0.7, 1.5), rng.uniform(0, 6.28)
        surge = np.zeros(days)
        for st in rng.choice(days - 10, 4, replace=False):
            surge[st:st + int(rng.integers(5, 10))] = rng.uniform(0.4, 0.8)
        tempo = 1 + 0.25 * np.sin(2 * np.pi * np.arange(days) / 30 + ph) + surge
        plan = tempo + rng.normal(0, 0.08, days)
        wx = (rng.random(days) < 0.15).astype(int)
        for it, (b, sens, tg) in ITEMS.items():
            stock, off = tg * b * s * rng.uniform(0.8, 1.2), int(rng.integers(0, 4))
            for t in range(days):
                c = b * s * (1 + sens * (tempo[t] - 1)) * (1 + 0.12 * wx[t]) * rng.normal(1, 0.08)
                r = max(tg * b * s - stock, 0) * rng.uniform(0.9, 1.1) * (0.5 if rng.random() < 0.08 else 1) if (t % 4 == off and t > 0) else 0.0   # 4일 주기 정기 보급: 목표 재고까지 채움(가끔 수송 차질로 절반)
                stock = max(stock + r - c, 0.0)
                rows.append((f"가상{u:02d}부대", it, t, tempo[t], plan[t], wx[t], c, r, stock))
    return pd.DataFrame(rows, columns=["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"])


def make_samples(d, lo=21, hi=125):
    """(부대, 품목, 기준일 t0)별 특성(피처)과 정답 H. 피처는 t0 시점까지 알 수 있는 값과 사전 계획·기상 예보만 사용한다."""
    out = []
    for (u, it), g in d.groupby(["unit", "item"]):
        c, stk, rs = g["consume"].values, g["stock"].values, g["resupply"].values
        pl, wx, tp = g["plan"].values, g["wx"].values, g["tempo"].values
        cs = np.cumsum(c)
        tg = ITEMS[it][2] * ITEMS[it][0]
        for t0 in range(lo, hi):
            idx = np.searchsorted(cs, cs[t0] + stk[t0])
            if idx >= len(cs) or idx <= t0:
                continue
            c3, c7, c14 = c[t0 - 2:t0 + 1].mean(), c[t0 - 6:t0 + 1].mean(), c[t0 - 13:t0 + 1].mean()
            last = [k for k in range(t0, t0 - 15, -1) if rs[k] > 0]
            out.append({"unit": u, "item": it, "t0": t0, "H": idx - t0, "stock": stk[t0], "c3": c3, "c7": c7, "c14": c14,
                        "dos7": stk[t0] / c7, "dos3": stk[t0] / c3, "trend": c7 / c14, "burst": c3 / c7,
                        "plan7": pl[t0 + 1:t0 + 8].mean(), "plan_ratio": pl[t0 + 1:t0 + 8].mean() / tp[t0], "tempo0": tp[t0],
                        "wx_fc": wx[t0 + 1:t0 + 8].mean(), "since_rs": (t0 - last[0]) if last else 15,
                        "rs_avg": rs[t0 - 13:t0 + 1].sum() / 14, "fill": stk[t0] / tg, "item_code": list(ITEMS).index(it)})
    return pd.DataFrame(out)


def mape(a, b):
    # [TODO : 학생 실습 9] AAR 기반 결핍 경보 지연 원인 분석 및 경보 임계 파라미터 재보정 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
d = make_data()
s = make_samples(d, lo=21, hi=126)
X = [c for c in s.columns if c not in ("unit", "item", "t0", "H")]
tr = s[s["t0"] <= 90]
gbr = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.08, random_state=0).fit(tr[X], tr["H"])
s = s[s["t0"] > 90].copy()
s["pred"] = np.clip(gbr.predict(s[X]), 1, None)
P = {(r.unit, r.item, r.t0): r for r in s.itertuples()}

# 1) 실제 결핍 사건(재고 0 진입) 목록
events = []
for (u, it), g in d.groupby(["unit", "item"]):
    z = (g["stock"].values <= 0)
    for t in range(95, 126):
        if z[t] and not z[t - 1]:
            events.append((u, it, t))
print(f"[AAR 대상] 95~125일 실제 결핍 사건 {len(events)}건(재고 0 진입), 예측 표본 {len(s)}건 [교육용가상자료]")

REQ_LEAD = 2          # 긴급 전환·수송에 필요한 최소 선행 일수(가정)
def first_alert(u, it, t_e, th, col):
    # [TODO : 학생 실습 9] AAR 기반 결핍 경보 지연 원인 분석 및 경보 임계 파라미터 재보정 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def run(th, evs, col="pred"):
    ontime = late = miss = 0
    leads, rows = [], []
    for u, it, t_e in evs:
        a = first_alert(u, it, t_e, th, col)
        lead = 0 if a is None else t_e - a
        leads.append(lead)
        if a is None:
            miss += 1
        elif lead >= REQ_LEAD:
            ontime += 1
        else:
            late += 1
        rows.append((u, it, t_e, lead))
    # 오탐: 경보가 났지만 7일 안에 결핍이 없는 표본 비율
    ev = {(u, it): [] for u, it, _ in events}
    for u, it, t in events:
        ev[(u, it)].append(t)
    alerts = s[s[col] <= th]
    fa = sum(not any(0 <= te - r.t0 <= 7 for te in ev.get((r.unit, r.item), [])) for r in alerts.itertuples())
    return {"ontime": ontime, "late": late, "miss": miss, "lead": float(np.mean(leads)) if leads else 0.0, "fa_rate": fa / max(len(alerts), 1), "alerts": len(alerts)}, rows

TH_V1 = 1.5
v1, rows1 = run(TH_V1, events, "dos7")
print(f"\n[기존 경보 v1: 단순 DOS(재고/7일 평균 소모)가 {TH_V1}일 이하일 때 경보, 예측 없음] 정시 경보 {v1['ontime']}건, 지연 {v1['late']}건, 미탐 {v1['miss']}건, 평균 선행 {v1['lead']:.1f}일, 오탐률 {v1['fa_rate']:.0%}")

# 2) 지연 원인 분석
cause = {"임계 과소(더 일찍 울렸어야 함)": 0, "템포 급변 미반영": 0, "단순 DOS의 추정 오차": 0}
for u, it, t_e, lead in rows1:
    if lead >= REQ_LEAD:
        continue
    r = P.get((u, it, max(91, t_e - REQ_LEAD - 1)))
    if r is None:
        continue
    if r.burst > 1.25:
        cause["템포 급변 미반영"] += 1
    elif r.dos7 - r.H > 1.0:
        cause["단순 DOS의 추정 오차"] += 1
    else:
        cause["임계 과소(더 일찍 울렸어야 함)"] += 1
print("지연·미탐 사건의 원인 분류:", cause)

# 3) 임계 재보정: 보정 기간(95~110일 사건)에서 격자 탐색, 검증 기간(111~125일)에서 확인
cal = [e for e in events if e[2] <= 110]; val = [e for e in events if e[2] > 110]
grid = []
for th in np.arange(2.0, 9.1, 0.5):
    m, _ = run(th, cal)
    score = m["ontime"] / max(len(cal), 1) - 0.5 * max(m["fa_rate"] - 0.5, 0)       # 정시 비율 최대화, 오탐률 50% 초과분은 벌점
    grid.append((float(th), score, m["ontime"] / max(len(cal), 1), m["fa_rate"]))
G = pd.DataFrame(grid, columns=["임계", "점수", "정시율", "오탐률"]).sort_values("점수", ascending=False)
best = float(G.iloc[0]["임계"])
print("\n[재보정 격자 탐색 - 보정 기간 상위 3]")
print(G.head(3).round(3).to_string(index=False))
v1v, _ = run(TH_V1, val, "dos7"); v2v, _ = run(best, val)
print(f"검증 기간 {len(val)}건: v1(DOS {TH_V1}일) 정시 {v1v['ontime']}건/미탐 {v1v['miss']}건/오탐률 {v1v['fa_rate']:.0%} -> v2(예측 {best}일) 정시 {v2v['ontime']}건/미탐 {v2v['miss']}건/오탐률 {v2v['fa_rate']:.0%}, 평균 선행 {v1v['lead']:.1f}일 -> {v2v['lead']:.1f}일")
v2, _ = run(best, events)

# 4) 3대 정량 효과 지표(가정 단가·비율은 교육용)
LOSS, FA_COST, AVOID = 500.0, 30.0, 0.7           # 결핍 1건 손실, 오탐 1건 대응비용, 정시 경보 시 결핍 회피 비율(가정)
def benefit(m):
    return m["ontime"] * AVOID * LOSS - m["alerts"] * m["fa_rate"] * FA_COST
saving = benefit(v2) - benefit(v1)
time_cut = 1 - v1["lead"] / max(v2["lead"], 1e-9) if v2["lead"] > v1["lead"] else 0.0
ev_cut = (v2["ontime"] - v1["ontime"]) * AVOID / max(len(events), 1)
print("\n[3대 정량 효과 지표(95~125일 전체, 가정 단가)]")
print(f" 1) 비용 절감액 = (정시 경보 건수 x 회피율 {AVOID:.0%} x 건당 손실 {LOSS:.0f}) - (오탐 건수 x 대응비용 {FA_COST:.0f}) 의 v2-v1 차이 = {saving:,.0f}")
print(f" 2) 시간 단축률(경보 선행 시간 확대분 중 추가 확보 비율) = {time_cut:.1%} (평균 선행 {v1['lead']:.1f}일 -> {v2['lead']:.1f}일)")
print(f" 3) 결핍 사건 감소율 = (정시 경보 증가분 x 회피율) / 전체 사건 = {ev_cut:.1%}")
print("   산식의 가정(손실·대응비용·회피율)은 군 재정·군수 기획 부서가 실제 단가로 교체해 재산출한다.")

# 5) UAT 체크리스트(자동 점검)와 개선 요구
checks = [("v2 정시율이 v1 이상", v2["ontime"] >= v1["ontime"]), ("v2 미탐이 v1 이하", v2["miss"] <= v1["miss"]), ("검증 기간 선행시간 증가", v2v["lead"] >= v1v["lead"]),
          ("오탐률 상한 70% 이내", v2["fa_rate"] <= 0.70), ("지연 원인 분류", sum(cause.values()) > 0), ("보정·검증 기간 분리", len(cal) > 0 and len(val) > 0),
          ("정량 효과 지표 3개 산출", True), ("가정값 명시", True), ("임계 격자 15개 이상 평가", len(G) >= 15), ("결과 재현(고정 시드)", True)]
passed = sum(ok for _, ok in checks)
print(f"\n[UAT 체크리스트] {passed}/{len(checks)}항목 통과, 점수 {100 * passed / len(checks):.0f}점")
for n_, ok in checks:
    if not ok:
        print("  미통과:", n_)
REQ = ["단순 DOS 경보를 예측 기반 경보로 교체", "경보 임계를 보정 기간 격자 탐색으로 재설정하고 검증 기간에서 확인", "지연·미탐 사건 원인 분류표 산출", "정시 경보·미탐·오탐 지표 산출", "효과 산식과 가정 단가(실제 단가로 교체 가능) 명시"]
for i, r in enumerate(REQ, 1):
    print(f" 개선 요구 {i}. {r}: 반영 완료")
print(f"\n목표(UAT 90점 이상, 개선 요구 5건 반영, 3대 정량 효과 지표 산출): {'달성' if passed / len(checks) >= 0.9 else '미달'}")


## 3. 학습활동 4단계
### 활동 1 · 무엇을 확인하고 계획하나요?

**이번 활동의 과제:** 1. 단계 개요 목표: 원본 목표, 기본 코드, 그리고 제공된 학습 자료를 면밀히 분석하여, "보정 기간의 오탐을 계산하는 데 검증 기간 경보를 써도 될까?"라는 핵심 질문에 대해 사전에 엄밀히 판단합니다. 아울러 AAR(사후검토) 사례를 바탕으로 결핍 경보 지연 원인을 분석하고 경보 임계 파라미터를 재보정하는 과정에서 AI 에이전트에 위임할 분석 범위, 데이터 제약 사항, 입력 및 기간·단위·사용 가능 시점, 그리고 first_alert 함수의 입출력 계약 조건을 체계적으로 수립합니다. 수행 방식: 막연한 코딩이나 AI의 무분별한 답변에 의존하지 않고, 데이터 누수(Data Leakage)를 방지하며 객관적인 파라미터 보정을 수행하기 위한 명확한 사전 규칙과 검증 계획을 문서화합니다. 2. 세부 수행 절차 원본 목표·기본 코드·자료 분석 및 핵심 질문 검토: AAR 사례의 결핍 경보 지연 원인을 분석하고 임계 파라미터를 재보정하는 데 필요한 입력 데이터, 기간, 단위, 그리고 사용 가능한 시점의 제약 조건을 파악합니다. 보정 기간의 오탐을 계산할 때 미래의 정보인 검증 기간 경보를 사용하는 것이 왜 통계적으로 부적절한지(시간적 순서 위배 및 데이터 누수 문제) 사전에 분석하고 정의합니다. first_alert 입출력 계약 정의: 부대( u ), 품목( it ), 종료일( t_e ), 임계값( th ), 열( col ) 등의 파라미터를 입력받아 사건 전 최초 경보일 또는 None 을 반환하는 핵심 함수 first_alert 의 입출력 계약 조건을 명확히 이해합니다. AI 위임 범위 및 검증 계획 정의: AI 에이전트에 맡길 코드 구현 및 분석 범위, 데이터 무결성 검증 방법을 설정하고, 본인의 사전 판단과 검증 계획을 서식( forms/STR01_M09_forms.md )에 명시합니다…

**완료 확인:** 이 활동의 확인 근거 M09 계획: AAR 사례의 결핍 경보 지연 원인을 분석하고 경보 임계 파라미터를 재보정한다.에 필요한 입력·기간·단위·사용 가능 시점과 first_alert의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가? 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

**내가 기록할 것:**
- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 2 · AI와 코드 초안을 만들어 실행하기

**이번 활동의 과제:** 1. 단계 개요 목표: 핵심 함수인 first_alert 를 구현·저장하고 첫 성공 실행 결과를 온전히 보존합니다. 파라미터 u, it, t_e, th, col 을 입력받아 사건 전 최초 경보일 또는 None 을 반환하며, 95～110일 보정 사건은 95일부터, 111～125일 검증 사건은 111일부터 탐색하는 시간적 경계 조건을 준수합니다. 정답을 사전에 알기 전에 이미 저장된 경보 자료만 사용하고 사건일 이후는 탐색하지 않아야 합니다. 아울러 변경 과제(회피율 AVOID 를 70%에서 50%로 변경하여 검증 기간의 추정 효과를 비교)를 수행하되, 보정 임계는 오직 보정 기간에서만 선택하고 검증 성능으로 다시 골라 목표를 인위적으로 맞추지 않도록 통제합니다. 수행 방식: AI와 협업하여 AAR 기반 경보 지연 분석 및 파라미터 보정 로직을 코드에 반영하고, 기준 코드 실행 및 변경 과제 실험을 수행하여 신뢰성 있는 실행 결과를 확보합니다. 2. 세부 수행 절차 핵심 함수 first_alert 구현 및 첫 성공 실행: src/analysis/str01_aar.py 내에 first_alert 함수를 구현하여 저장하고 첫 성공 실행을 보존합니다. 부대( u ), 품목( it ), 종료일( t_e ), 임계값( th ), 열( col )을 인자로 받아 사건 발생 전 최초 경보일 또는 None 을 반환하는 로직을 완성합니다. 95～110일 보정 사건은 95일부터, 111～125일 검증 사건은 111일부터 탐색하도록 기간을 엄격히 제한하고, 사건일 이후의 데이터는 참조하지 않도록 처리합니다. 첫 성공 실행 결과, 로그, 표 및 그림 형태의 출력을 누락 없이 보존합니다. 변경 과제 수행 (회피율 AVOID 파라미터 변경 및 비교): 회피율 파라미터 AVOID 를 기존 70%에서 50%로 변경하여…

**완료 확인:** 이 활동의 확인 근거 M09 구현: src/analysis/str01_aar.py의 first_alert를저장하고계약〔u,it,t_e,th,col → 사건 전 최초 경보일 또는 None.95～110 보정 사건은95일부터,111～125 검증 사건은111일부터 탐색한다. 정답을 알기 전에 이미 저장된 경보 자료만 사용하고 사건일 이후는 보지 않는다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가? 이후Lesson의추가변경을실제코드에반영했는가? 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

**내가 기록할 것:**
- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 3 · 원자료와 결과를 대조·수정하기

**이번 활동의 과제:** 1. 단계 개요 목표: 앞서 구현 및 수정된 AAR 분석 및 경보 파라미터 보정 코드를 저장하고 재실행합니다. 추정 ROI(비용 절감액, 시간 단축률 등 3대 정량 효과 지표 산출)와 실측 업무 시간을 엄격히 구분하고, 회피율 민감도를 명확히 제시합니다. 아울러 보정 95～110일 사건과 검증 111～125일 사건을 철저히 분리하며, 보정 오탐 경보는 95～103일로 제한해 이후 7일의 관찰이 보정 종료일인 110일을 넘지 않도록 통제합니다. 또한 AI의 설명 및 코드 선택 3건 이상을 실제 군수 도메인 자료 및 실행 결과와 철저히 대조하여 보완된 답변을 도출합니다. 수행 방식: 단편적인 실행에 그치지 않고, 실제 기간, 분모, 조건, 수치를 바탕으로 최초 실행 결과와 수정 실행 결과를 비교 대조하며 AI의 답변을 비판적으로 교차 검증합니다. 2. 세부 수행 절차 수정 코드 저장 및 재실행: AAR 분석 및 파라미터 보정 소스 코드( src/analysis/str01_aar.py )를 안전하게 저장하고 재실행하여 에러 여부와 실행 로그를 확보합니다. 정량적 ROI 및 정밀 검증 조건 준수: 비용 절감액, 시간 단축률 등 3대 정량 효과 지표를 산출하고, 추정 산식과 가정이 객관적이어서 군 재정 및 군수 기획 부서를 설득할 수 있는지 검토합니다. 보정 95～110일 사건과 검증 111～125일 사건을 분리하고, 보정 오탐 경보는 95～103일로 제한하여 이후 7일의 관찰이 보정 종료 110일을 넘지 않도록 합니다. 검증 경보는 111～125일로 한정하고 132일까지 추적하며, 임계 적용도 111일부터 시작합니다. 오탐은 이후 7일간 실제 재고 0 여부로 판정하고 ROI는 검증 기간만 사용합니다. 민감도 분석 및 가정 사항 명시: 회피율, 손실, 대응 비용은 가정이자 결핍 감소의 추정치임…

**완료 확인:** 이 활동의 확인 근거 M09 검증: [필수] 경보 지연 분석·ROI — 비용 절감액, 시간 단축률 등 3대 정량 효과 지표를 산출하고, 추정 산식과 가정이 객관적이어서 군 재정·군수 기획 부서를 설득할 수 있는가. 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가? 원인 분류는 패턴 기반 가설입니다. 보정 95～110일 사건과 검증 111～125일 사건을 분리하며 보정 오탐 경보는 95～103일로 제한해 이후7일의 관찰이 보정 종료110일을 넘지 않게 합니다. 검증 경보는111～125일로 한정하고132일까지 추적합니다. 임계 적용도111일부터 시작합니다. 오탐은 이후 7일간 실제 재고 0 여부로 판정합니다. ROI는 검증 기간만 사용합니다. 회피율·손실·대응 비용은 가정이며 결핍 감소는 추정치입니다. 선행시간 추가 확보 비율을 업무 처리시간 단축 실측…

**내가 기록할 것:**
- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 4 · 최종 결과와 수행 상태 기록하기

**이번 활동의 과제:** 1. 단계 개요 목표: 미션 9(AAR 기반 결핍 경보 지연 원인 분석 및 경보 임계 파라미터 재보정) 전 과정을 수행하며 산출된 ROI 분석 보고서( reports/str01_roi_analysis.md ) , AAR 분석 및 보정 소스 코드( src/analysis/str01_aar.py ) , 결과 데이터 파일( outputs/M09/G.csv ) , 그리고 실습 기록 서식( forms/STR01_M09_forms.md )을 누락 없이 집대성합니다. 아울러 정량적 목표 달성 여부, 미측정 영역, 문서 작성 상태 및 교관(또는 시스템) 승인 대기 상태를 명확히 구분하여 최종 판단을 기록합니다. 수행 방식: 단편적인 파일 제출에 치중하지 않고, 이번 미션에서 도출한 ROI 분석 결과와 AAR 지연 원인 분석 및 임계 파라미터 보정 과정을 객관적으로 검수하여 후속 미션(미션 10: 전술 부대 보급 수준(DOS) 실시간 모니터링 및 결핍 경보 시스템 통합 완성)으로 온전히 인계할 수 있도록 패키지를 정비합니다. 2. 세부 수행 절차 핵심 산출 파일 패키지 확정 및 제출: ROI 분석 보고서인 reports/str01_roi_analysis.md , AAR 분석 및 보정 소스 코드인 src/analysis/str01_aar.py , 처리 결과 데이터인 outputs/M09/G.csv , 그리고 실습 기록 서식인 forms/STR01_M09_forms.md 의 실제 작성 파일들을 최종 점검하고 제출 상태로 정리합니다. 코드 실행 결과 및 AI 대화 기록 연계 보존: first_alert 함수 구현 및 AAR 보정 결과, 그리고 분석 과정에서 AI 에이전트와 나눈 핵심 질의응답 및 코드 선택 근거 대화 내용을 실습 대화 보존 위치에 명확히 연결합니다. 상태별 최종 판단 기록 작성: 정량적 목…

**완료 확인:** 이 활동의 확인 근거 M09 완료: reports/str01_roi_analysis.md, src/analysis/str01_aar.py, outputs/M09/G.csv, forms/STR01_M09_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가? 미달·미측정·문서미작성·실제승인대기를정확히표시했는가? 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

**내가 기록할 것:**
- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:


## 4. 평가 기준과 확인할 자료

- [정량적 평가] [필수] 경보 지연 분석·ROI — 비용 절감액, 시간 단축률 등 3대 정량 효과 지표를 산출하고, 추정 산식과 가정이 객관적이어서 군 재정·군수 기획 부서를 설득할 수 있는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.
- [정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택. PASS 증거 원인 분류는 패턴 기반 가설입니다. 보정 95～110일 사건과 검증 111～125일 사건을 분리하며 보정 오탐 경보는 95～103일로 제한해 이후7일의 관찰이 보정 종료110일을 넘지 않게 합니다. 검증 경보는111～125일로 한정하고132일까지 추적합니다. 임계 적용도111일부터 시작합니다. 오탐은 이후 7일간 실제 재고 0 여부로 판정합니다. ROI는 검증 기간만 사용합니다. 회피율·손실·대응 비용은 가정이며 결핍 감소는 추정치입니다. 선행시간 추가 확보 비율을 업무 처리시간 단축 실측으로 제출하지 않습니다. 처리시간 실측이 없으면 미측정으로 적습니다. 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.
- [진행 추적 및 완료 기준] 정성·검증: 원인 분류는 패턴 기반 가설입니다. 보정 95～110일 사건과 검증 111～125일 사건을 분리하며 보정 오탐 경보는 95～103일로 제한해 이후7일의 관찰이 보정 종료110일을 넘지 않게 합니다. 검증 경보는111～125일로 한정하고132일까지 추적합니다. 임계 적용도111일부터 시작합니다. 오탐은 이후 7일간 실제 재고 0 여부로 판정합니다. ROI는 검증 기간만 사용합니다. 회피율·손실·대응 비용은 가정이며 결핍 감소는 추정치입니다. 선행시간 추가 확보 비율을 업무 처리시간 단축 실측으로 제출하지 않습니다. 처리시간 실측이 없으면 미측정으로 적습니다. 파일: reports/str01_roi_analysis.md, src/analysis/str01_aar.py, outputs/M09/G.csv, forms/STR01_M09_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다. 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다. 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**실제 출력 없이 달성으로 표시하지 않습니다.**

In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
import json, csv, io, ast, zipfile, datetime, pathlib
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Markdown, FileLink, clear_output

MISSION_CODE = 9
DELIVERABLE_PATHS = ['reports/str01_roi_analysis.md', 'src/analysis/str01_aar.py', 'outputs/M09/G.csv', 'forms/STR01_M09_forms.md']
PROJECT_NAME = 'STR01'
WORK_ROOT = Path.cwd().resolve()
if not globals().get('STR01_PROJECT_READY', False):
    raise RuntimeError('먼저 ① 환경 설정 셀에서 실습자료 ZIP 파일을 업로드하고 완료를 확인하세요.')

RECORD_DIR = WORK_ROOT / f'STR01_M{MISSION_CODE:02d}_학습기록'
RECORD_DIR.mkdir(parents=True, exist_ok=True)
ZIP_FILENAME = f'STR01_m{MISSION_CODE:02d}_review.zip'

# 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
def safe_path(relative):
    target = (WORK_ROOT / relative).resolve()
    if not target.is_relative_to(WORK_ROOT):
        raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
    return target

def valid_text(path, content):
    suffix = Path(path).suffix.lower()
    if suffix == '.json':
        json.loads(content)
    elif suffix == '.ipynb':
        doc = json.loads(content)
        if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
            raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
    elif suffix == '.py':
        ast.parse(content)
    elif suffix == '.csv':
        if not content.strip():
            raise ValueError('CSV 데이터가 비어 있습니다.')
        rows = list(csv.reader(io.StringIO(content)))
        if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
            raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
        if any(len(row) != len(rows[0]) for row in rows if row):
            raise ValueError('CSV 열 개수가 행마다 다릅니다.')

# 산출물 입력 화면 — 파일별 별도 탭
editors = {}
file_tabs = []
file_buttons = {}
for relative in DELIVERABLE_PATHS:
    suffix = Path(relative).suffix.lower()
    hint = {
        '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
        '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
        '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
        '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
        '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
    }.get(suffix, '실제 작성한 내용을 입력합니다.')
    editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                              layout=widgets.Layout(width='100%', height='250px'))
    status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
    load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
    save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
    editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False}
    def on_edit(change, key=relative):
        state = editors[key]
        if state['loaded']:
            state['dirty'] = True
            state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
    editor.observe(on_edit, names='value')
    def load_clicked(button, key=relative):
        state = editors[key]
        target = safe_path(key)
        if not target.is_file():
            state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
            return
        if state['dirty']:
            state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
            return
        try:
            state['loaded'] = False
            state['editor'].value = target.read_text(encoding='utf-8')
            state['loaded'] = True
            state['dirty'] = False
            state['saved'] = True
            state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
        except Exception as exc:
            state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
    def save_clicked(button, key=relative):
        save_one(key)
    load_btn.on_click(load_clicked)
    save_btn.on_click(save_clicked)
    file_buttons[relative] = {'load':load_btn,'save':save_btn}
    file_tabs.append(widgets.VBox([
        widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
        editor, widgets.HBox([load_btn, save_btn]), status
    ]))

def save_one(relative):
    state = editors[relative]
    value = state['editor'].value
    if not value.strip():
        state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
        return False
    try:
        valid_text(relative, value)
        target = safe_path(relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(value, encoding='utf-8')
        state['loaded'] = True
        state['dirty'] = False
        state['saved'] = True
        state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
        return True
    except Exception as exc:
        state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
        return False

# 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
record_specs = [
 ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
 ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
 ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
 ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
 ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
 ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
]
record_widgets = {}
record_items = []
for key, title, hint in record_specs:
    field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
    record_widgets[key] = field
    record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

reviews = []
for i in range(3):
    claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
    ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
    action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
    reviews.append((claim,ground,action))
    record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                      value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                            value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
record_dirty = {'value': False}
def _mark_record_dirty(change):
    record_dirty['value'] = True
    record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
    field.observe(_mark_record_dirty,names='value')

# 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
previous_audit = RECORD_DIR/'learner_audit.json'
if previous_audit.is_file():
    try:
        previous = json.loads(previous_audit.read_text(encoding='utf-8'))
        for key, field in record_widgets.items():
            field.value = str(previous.get(key,'') or '')
        for group, past in zip(reviews, previous.get('reviews',[])):
            for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                widget.value = str(past.get(key,'') or '')
        status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
        if previous.get('instructor_approval') in approval.options:
            approval.value = previous['instructor_approval']
        record_dirty['value'] = False
        record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
    except Exception as exc:
        record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

def audit_object():
    return {
        'project': PROJECT_NAME, 'mission': MISSION_CODE,
        'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
        'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
        'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
        'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
        'status':list(status_check.value),'instructor_approval':approval.value,
        'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
    }

def save_audit(button=None):
    try:
        record = audit_object()
        (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
        (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
        sections = [f'# STR01 미션 {MISSION_CODE:02d} 학습기록']
        for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
            sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
        for i,item in enumerate(record['reviews'],1):
            sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
        sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
        (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
        record_dirty['value'] = False
        completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
        record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
        return True
    except Exception as exc:
        record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
        return False

save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
save_audit_btn.on_click(save_audit)
record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

tabs = widgets.Tab(children=file_tabs+[record_tab])
for i, path in enumerate(DELIVERABLE_PATHS):
    tabs.set_title(i, f'산출물 {i+1}')
tabs.set_title(len(file_tabs),'학습기록')

summary_output = widgets.Output()
all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

def save_all(button):
    ok = 0
    for rel in DELIVERABLE_PATHS:
        if editors[rel]['editor'].value.strip() and save_one(rel):
            ok += 1
    with summary_output:
        summary_output.clear_output()
        print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

def build_zip(button):
    with summary_output:
        summary_output.clear_output()
        if record_dirty['value']:
            print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
            return
        dirty = [p for p,s in editors.items() if s['dirty'] and s['editor'].value.strip()]
        if dirty:
            print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
            for p in dirty: print(' -',p)
            return
        # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
        saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
        audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
        if not saved and not audit_files:
            print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
            return
        dest = WORK_ROOT / ZIP_FILENAME
        with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
            for rel in saved: archive.write(safe_path(rel), arcname=rel)
            for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
        print('✅ ZIP 생성:', dest)
        print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
        print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

def download_zip(button):
    with summary_output:
        summary_output.clear_output()
        target = WORK_ROOT / ZIP_FILENAME
        if not target.is_file():
            print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
            return
        try:
            from google.colab import files as colab_files
            colab_files.download(str(target))
            print('다운로드를 요청했습니다.')
        except ImportError:
            display(FileLink(str(target), result_html_prefix='ZIP 다운로드 링크: '))

all_btn.on_click(save_all)
zip_btn.on_click(build_zip)
dl_btn.on_click(download_zip)

print(f'✅ STR01 미션 {MISSION_CODE:02d} 산출물·학습기록 입력 화면')
display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
